# 16 · Causal operator features for turbofan RUL

On NASA's C-MAPSS turbofan data the task is remaining-useful-life (RUL)
prediction. The omnibias approach builds **causal** closed-form operator features
from each engine's sensor trajectory — derivatives, differences, rolling
summaries — that never peek at future cycles. This notebook demonstrates the
feature construction on a synthetic degradation trajectory (data-free); the full
FD001 benchmark is in `examples/symbolic_discovery/cmapss_feature_discovery/`.

In [ ]:
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, ".")
sys.path.insert(0, "..")
from _style import set_style, PRIMARY, ACCENT, GOOD
set_style()

from examples.symbolic_discovery.cmapss_feature_discovery.benchmark import _past_trajectory_matrix

## A synthetic sensor trajectory

One engine, one sensor that drifts and accelerates toward failure.

In [ ]:
rng = np.random.default_rng(0)
n = 120
t = np.arange(1, n + 1)
signal = 10.0 + 0.02 * t + 8.0 * (t / n) ** 3  # slow drift + late acceleration
s_2 = signal + rng.normal(0.0, 0.15, size=n)
df = pd.DataFrame({"unit_number": np.ones(n, int), "time_cycles": t, "s_2": s_2})
matrix, names = _past_trajectory_matrix(df, ["time_cycles", "s_2"])
print("operator feature columns (sample):", [nm for nm in names if nm.startswith("s_2")][:8])

## Causal derivative features track the degradation rate

The first-difference operator `s_2_diff1` rises as the sensor accelerates — a
leading indicator of failure — and is computed using only past rows.

In [ ]:
diff1 = matrix[:, names.index("s_2_diff1")]
roll = matrix[:, names.index("s_2_roll5")] if "s_2_roll5" in names else None
fig, (axl, axr) = plt.subplots(1, 2, figsize=(11, 4.0))
axl.plot(t, s_2, color=PRIMARY, label="sensor s_2")
if roll is not None:
    axl.plot(t, roll, color=GOOD, lw=1.6, label="causal rolling mean")
axl.set_title("Sensor trajectory"); axl.set_xlabel("cycle"); axl.legend()
axr.plot(t, diff1, color=ACCENT)
axr.set_title("Causal first-difference operator (degradation rate)"); axr.set_xlabel("cycle")
plt.tight_layout()

## Takeaway

Closed-form, leak-free operator features turn raw sensor streams into
interpretable degradation signals. On real FD001 these features feed a sparse
selector that improves RUL accuracy over raw-sensor baselines — run
`python -m examples.symbolic_discovery.cmapss_feature_discovery.run_demo` after
downloading the dataset.